In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

# Load datasets
sales_df = pd.read_csv('/content/sales_train_validation.csv')
calendar_df = pd.read_csv('/content/calendar.csv')
prices_df = pd.read_csv('/content/sell_prices.csv')

# Basic exploration
print(f"Sales data shape: {sales_df.shape}")
print(f"Date range: {calendar_df['date'].min()} to {calendar_df['date'].max()}")
print(f"Stores: {sales_df['store_id'].unique()}")
print(f"Categories: {sales_df['cat_id'].unique()}")

Sales data shape: (30490, 1919)
Date range: 2011-01-29 to 2016-06-19
Stores: ['CA_1' 'CA_2' 'CA_3' 'CA_4' 'TX_1' 'TX_2' 'TX_3' 'WI_1' 'WI_2' 'WI_3']
Categories: ['HOBBIES' 'HOUSEHOLD' 'FOODS']


In [2]:
def create_rolling_features(sales_data, windows=[7, 14, 28]):
    """Create rolling statistical features"""
    features = {}

    for window in windows:
        features[f'rolling_mean_{window}'] = sales_data.rolling(window).mean()
        features[f'rolling_std_{window}'] = sales_data.rolling(window).std()
        features[f'rolling_cv_{window}'] = features[f'rolling_std_{window}'] / features[f'rolling_mean_{window}']

        # Z-score for anomaly detection
        features[f'zscore_{window}'] = (sales_data - features[f'rolling_mean_{window}']) / features[f'rolling_std_{window}']

    return pd.DataFrame(features)

In [3]:
def engineer_calendar_features(calendar_df):
    """Extract meaningful calendar features"""
    calendar_features = calendar_df.copy()

    # Convert date
    calendar_features['date'] = pd.to_datetime(calendar_features['date'])

    # Time-based features
    calendar_features['day_of_week'] = calendar_features['date'].dt.dayofweek
    calendar_features['month'] = calendar_features['date'].dt.month
    calendar_features['quarter'] = calendar_features['date'].dt.quarter
    calendar_features['is_weekend'] = calendar_features['day_of_week'].isin([5, 6]).astype(int)

    # Event clustering
    calendar_features['has_event'] = (~calendar_features['event_name_1'].isna()).astype(int)
    calendar_features['event_count'] = (
        (~calendar_features['event_name_1'].isna()).astype(int) +
        (~calendar_features['event_name_2'].isna()).astype(int)
    )

    # SNAP benefits (potential demand driver)
    snap_cols = ['snap_CA', 'snap_TX', 'snap_WI']
    calendar_features['snap_total'] = calendar_features[snap_cols].sum(axis=1)

    return calendar_features

In [4]:
def detect_price_changes(prices_df):
    """Identify significant price changes that might affect sales"""
    price_features = prices_df.copy()

    # Sort by store, item, and week
    price_features = price_features.sort_values(['store_id', 'item_id', 'wm_yr_wk'])

    # Calculate price changes
    price_features['price_change'] = price_features.groupby(['store_id', 'item_id'])['sell_price'].pct_change()
    price_features['price_change_abs'] = price_features['price_change'].abs()

    # Flag significant price changes (>10% change)
    price_features['significant_price_change'] = (price_features['price_change_abs'] > 0.1).astype(int)

    return price_features

In [5]:
def identify_disruptions(sales_series, calendar_features, price_features,
                        zscore_threshold=-2.0, min_duration=2):
    """
    Identify potential supply chain disruptions

    Disruption criteria:
    1. Significant drop in sales (z-score < threshold)
    2. Not explained by price changes or major events
    3. Sustained for minimum duration
    """

    disruptions = []

    # Calculate anomaly scores
    rolling_mean = sales_series.rolling(14).mean()
    rolling_std = sales_series.rolling(14).std()
    zscore = (sales_series - rolling_mean) / rolling_std

    # Identify potential disruption periods
    anomaly_mask = zscore < zscore_threshold

    # Filter out event-driven drops
    event_mask = calendar_features['has_event'] == 1
    price_change_mask = price_features['significant_price_change'] == 1

    # True disruptions: anomalies not explained by events or price changes
    disruption_mask = anomaly_mask & ~event_mask & ~price_change_mask

    # Find consecutive disruption periods
    disruption_periods = find_consecutive_periods(disruption_mask, min_duration)

    return disruption_periods, disruption_mask

def find_consecutive_periods(mask, min_duration=2):
    """Find consecutive True periods in boolean mask"""
    periods = []
    start = None

    for i, val in enumerate(mask):
        if val and start is None:
            start = i
        elif not val and start is not None:
            if i - start >= min_duration:
                periods.append((start, i-1))
            start = None

    return periods

In [6]:
def classify_disruption_severity(sales_drop_pct, duration_days):
    """Classify disruption severity based on impact and duration"""

    if sales_drop_pct < -50 or duration_days > 14:
        return 'Critical'
    elif sales_drop_pct < -30 or duration_days > 7:
        return 'High'
    elif sales_drop_pct < -15 or duration_days > 3:
        return 'Medium'
    else:
        return 'Low'

In [7]:
def create_time_aware_splits(data, train_ratio=0.7, val_ratio=0.15):
    """Create chronological splits to avoid data leakage"""

    n_samples = len(data)
    train_end = int(n_samples * train_ratio)
    val_end = int(n_samples * (train_ratio + val_ratio))

    train_data = data.iloc[:train_end]
    val_data = data.iloc[train_end:val_end]
    test_data = data.iloc[val_end:]

    return train_data, val_data, test_data

In [11]:
!pip install catboost

!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 19.9 MB/s eta 0:00:00


In [12]:
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix
import optuna

def train_disruption_models(X_train, y_train, X_val, y_val):
    """Train and compare different models"""

    models = {}

    # Random Forest
    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=10,
        class_weight='balanced',
        random_state=42
    )
    rf_model.fit(X_train, y_train)
    models['RandomForest'] = rf_model

    # CatBoost
    categorical_features = ['store_id', 'cat_id', 'dept_id', 'day_of_week', 'month']

    cb_model = CatBoostClassifier(
        iterations=500,
        depth=8,
        learning_rate=0.1,
        class_weights='Balanced',
        random_seed=42,
        verbose=False
    )
    cb_model.fit(X_train, y_train,
                 cat_features=categorical_features,
                 eval_set=(X_val, y_val),
                 early_stopping_rounds=50)
    models['CatBoost'] = cb_model

    return models

def hyperparameter_optimization(X_train, y_train, X_val, y_val):
    """Optimize hyperparameters using Optuna"""

    def objective(trial):
        # Define hyperparameter search space
        params = {
            'iterations': trial.suggest_int('iterations', 100, 1000),
            'depth': trial.suggest_int('depth', 4, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        }

        model = CatBoostClassifier(**params, random_seed=42, verbose=False)
        model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

        preds = model.predict(X_val)
        accuracy = (preds == y_val).mean()

        return accuracy

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=100)

    return study.best_params

In [13]:
def calculate_business_metrics(y_true, y_pred, y_prob, cost_matrix=None):
    """Calculate business-relevant metrics"""

    from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

    # Standard metrics
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    auc = roc_auc_score(y_true, y_prob)

    # Business metrics
    false_positive_rate = np.sum((y_pred == 1) & (y_true == 0)) / np.sum(y_true == 0)
    false_negative_rate = np.sum((y_pred == 0) & (y_true == 1)) / np.sum(y_true == 1)

    # Early warning capability (predict disruptions X days in advance)
    early_warning_accuracy = calculate_early_warning_accuracy(y_true, y_pred)

    metrics = {
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'auc': auc,
        'false_positive_rate': false_positive_rate,
        'false_negative_rate': false_negative_rate,
        'early_warning_accuracy': early_warning_accuracy
    }

    return metrics

def calculate_early_warning_accuracy(y_true, y_pred, lead_days=3):
    """Calculate accuracy of predictions made X days before disruption"""
    # Implementation depends on your time series structure
    pass

In [14]:
def analyze_feature_importance(model, feature_names, top_n=20):
    """Analyze and visualize feature importance"""

    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    elif hasattr(model, 'get_feature_importance'):
        importances = model.get_feature_importance()

    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)

    # Plot top features
    plt.figure(figsize=(10, 8))
    sns.barplot(data=importance_df.head(top_n), x='importance', y='feature')
    plt.title('Top Feature Importances for Supply Chain Disruption Prediction')
    plt.tight_layout()
    plt.show()

    return importance_df

In [15]:
import networkx as nx
import plotly.graph_objects as go

def create_supply_chain_graph(sales_df, disruption_data):
    """Create knowledge graph representing supply chain relationships"""

    G = nx.Graph()

    # Add nodes for different entities
    stores = sales_df['store_id'].unique()
    categories = sales_df['cat_id'].unique()
    departments = sales_df['dept_id'].unique()

    # Add store nodes
    for store in stores:
        G.add_node(f"store_{store}", type='store', id=store)

    # Add category nodes
    for cat in categories:
        G.add_node(f"cat_{cat}", type='category', id=cat)

    # Add department nodes
    for dept in departments:
        G.add_node(f"dept_{dept}", type='department', id=dept)

    # Add edges based on relationships
    for _, row in sales_df.iterrows():
        store_node = f"store_{row['store_id']}"
        cat_node = f"cat_{row['cat_id']}"
        dept_node = f"dept_{row['dept_id']}"

        G.add_edge(store_node, cat_node, relationship='sells')
        G.add_edge(store_node, dept_node, relationship='contains')
        G.add_edge(cat_node, dept_node, relationship='belongs_to')

    return G

def simulate_risk_propagation(graph, initial_disruptions, propagation_rate=0.3):
    """Simulate how disruptions propagate through the network"""

    risk_scores = {node: 0.0 for node in graph.nodes()}

    # Set initial risk scores for disrupted nodes
    for node in initial_disruptions:
        risk_scores[node] = 1.0

    # Propagate risk through network
    for _ in range(5):  # 5 iterations of propagation
        new_scores = risk_scores.copy()

        for node in graph.nodes():
            neighbors = list(graph.neighbors(node))
            if neighbors:
                neighbor_risk = np.mean([risk_scores[n] for n in neighbors])
                new_scores[node] = max(risk_scores[node],
                                     neighbor_risk * propagation_rate)

        risk_scores = new_scores

    return risk_scores

In [16]:
def visualize_supply_chain_network(graph, risk_scores=None):
    """Create interactive network visualization"""

    pos = nx.spring_layout(graph, k=1, iterations=50)

    edge_x = []
    edge_y = []
    for edge in graph.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(x=edge_x, y=edge_y,
                           line=dict(width=0.5, color='#888'),
                           hoverinfo='none',
                           mode='lines')

    node_x = []
    node_y = []
    node_text = []
    node_color = []

    for node in graph.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_text.append(node)

        if risk_scores:
            node_color.append(risk_scores[node])
        else:
            node_color.append(0)

    node_trace = go.Scatter(x=node_x, y=node_y,
                           mode='markers+text',
                           hoverinfo='text',
                           text=node_text,
                           textposition="middle center",
                           marker=dict(size=10,
                                     color=node_color,
                                     colorscale='Reds',
                                     showscale=True,
                                     colorbar=dict(title="Risk Score")))

    fig = go.Figure(data=[edge_trace, node_trace],
                   layout=go.Layout(
                    title='Supply Chain Risk Network',
                    titlefont_size=16,
                    showlegend=False,
                    hovermode='closest',
                    margin=dict(b=20,l=5,r=5,t=40),
                    annotations=[ dict(
                        text="Supply Chain Network Analysis",
                        showarrow=False,
                        xref="paper", yref="paper",
                        x=0.005, y=-0.002 ) ],
                    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))

    return fig